In [ ]:

import scanpy as sc, squidpy as sq, pandas as pd, numpy as np, seaborn as sns, matplotlib.pyplot as plt, anndata, scipy, harmonypy as hm, os, warnings, sys
if not sys.warnoptions:
    import warnings
    warnings.simplefilter("ignore")
    
import os
plot_directory = "/Users/aly/Downloads/2024_Ali_scRNAseq/2025_Spatial_Fibrosis/2025_Squidpy_Spatial/Human/Analysis_v4_Final"
os.makedirs(plot_directory, exist_ok=True)

In [ ]:
base = "/Users/aly/Downloads/2024_Ali_scRNAseq/2025_Spatial_Fibrosis/spatial-lung-fibrosis-master/data/human/hs_visium_spaceranger_output"
samples = sorted([d for d in os.listdir(base) if d.startswith("V10") or d.startswith("V19")])
adatas = []
for s in samples:
    path = os.path.join(base, s)
    ad = sc.read_visium(path, load_images=True)
    ad.var_names_make_unique()
    ad.obs["sample"] = s
    adatas.append(ad)

adata = adatas[0].concatenate(adatas[1:], batch_key="sample", batch_categories=samples)

meta = pd.read_csv("/Users/aly/Downloads/2024_Ali_scRNAseq/2025_Spatial_Fibrosis/spatial-lung-fibrosis-master/data/human/hs_visium_metadata.tsv", sep="\t").set_index("sample_id")
adata.obs["sample"] = adata.obs["sample"].astype(str)
adata.obs = adata.obs.join(meta, on="sample", rsuffix="_meta")

In [ ]:
print(adata.obs[["sample", "sample_name", "slide_id", "slide_ca", "condition", "subject_alias", "tissue_alias"]].drop_duplicates())

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("Mt-")
adata.var["rps"] = adata.var_names.str.startswith("RPS")
adata.var["rpl"] = adata.var_names.str.startswith("RPL")

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "rps", "rpl"], percent_top=None, log1p=False, inplace=True)

adata = adata[adata.obs.n_genes_by_counts > 200]
adata = adata[adata.obs.pct_counts_mt < 20]
adata = adata[adata.obs.pct_counts_rps < 30]
adata = adata[adata.obs.pct_counts_rpl < 30]

In [ ]:
adata

In [ ]:
print('Total number of cells: {:d}'.format(adata.n_obs))
sc.pp.filter_cells(adata, min_counts=1200)
print('Number of cells after min count filter: {:d}'.format(adata.n_obs))
sc.pp.filter_cells(adata, max_counts=40000)
print('Number of cells after max count filter: {:d}'.format(adata.n_obs))
adata = adata[adata.obs['pct_counts_mt'] < 20]
print('Number of cells after MT filter: {:d}'.format(adata.n_obs))
sc.pp.filter_cells(adata, min_genes=600)
print('Number of cells after gene filter: {:d}'.format(adata.n_obs))
print('Total number of genes: {:d}'.format(adata.n_vars))
sc.pp.filter_genes(adata, min_cells=20)
print('Number of genes after cell filter: {:d}'.format(adata.n_vars))

In [ ]:
sc.pp.filter_genes(adata, min_cells=3)
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, flavor="seurat", n_top_genes=22487)
adata.raw = adata.copy()
adata = adata[:, adata.var.highly_variable].copy()
sc.pp.scale(adata)
adata

In [ ]:
adata.uns["spatial"] = {s: adatas[i].uns["spatial"][s] for i, s in enumerate(samples)}

In [ ]:
sc.pp.pca(adata, n_comps=50)
ho = hm.run_harmony(adata.obsm['X_pca'], adata.obs, 'sample_name')
adata.obsm['X_pca_harmony'] = ho.Z_corr.T

In [ ]:
sc.pp.neighbors(adata, use_rep='X_pca_harmony')
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.4, key_added='leiden_0.4')

In [ ]:
print(adata.obs.columns)
print(adata.obs["sample"].unique())
print(adata.obs["sample_name"].unique() if "sample_name" in adata.obs.columns else "No sample_name column")

In [ ]:
order = [
    'HC_1.TH010.B0.1', 'HC_1.TH010.B0.2', 'HC_2.TH020.B0.1', 'HC_3.TH030.B0.1', 'HC_3.TH030.B0.2', 'HC_4.TH040.B0.2',
    'IPF_1.TD011.B1.1', 'IPF_1.TD011.B1.2', 'IPF_1.TD012.B2.1', 'IPF_1.TD012.B2.2', 'IPF_1.TD013.B3.1', 'IPF_1.TD013.B3.2',
    'IPF_2.TD021A.B1.1', 'IPF_2.TD021A.B1.2', 'IPF_2.TD021B.B2.1', 'IPF_2.TD021B.B2.2', 'IPF_2.TD022.B3.1', 'IPF_2.TD022.B3.2',
    'IPF_3.TD031.B1.2', 'IPF_3.TD032A.B2.2', 'IPF_3.TD032B.B3.2',
    'IPF_4.TD041A.B1.1', 'IPF_4.TD041A.B1.2', 'IPF_4.TD041B.B2.1', 'IPF_4.TD042.B3.1'
]

In [ ]:
for res in [0.6, 0.8, 1.0]: sc.tl.leiden(adata, resolution=res, key_added=f'leiden_{res}')

In [ ]:
sc.pl.umap(adata, color=["leiden_0.4", "leiden_0.6", "leiden_0.8", "leiden_1.0"], ncols=2, size=15)#, legend_loc="on data")

In [ ]:
import matplotlib.pyplot as plt
import squidpy as sq

samples = ["HC_3.TH030.B0.2", "IPF_1.TD013.B3.2", "IPF_2.TD021B.B2.1", "IPF_3.TD032B.B3.2", "IPF_4.TD042.B3.1"]
fig, axs = plt.subplots(1, 5, figsize=(25, 5))

for i, sample_name in enumerate(samples):
    ad = adata[adata.obs["sample_name"] == sample_name].copy()
    sq.pl.spatial_scatter(ad, color="leiden_0.6", library_id=ad.obs["sample"].iloc[0], size=1.6, ax=axs[i])
    axs[i].set_title(sample_name)

plt.tight_layout()
plt.show()

In [ ]:
adata.obs.groupby(['sample_name', 'leiden_0.4']).size().unstack(fill_value=0)

In [ ]:
order = [
    'HC_1.TH010.B0.1', 'HC_1.TH010.B0.2', 'HC_2.TH020.B0.1', 'HC_3.TH030.B0.1', 'HC_3.TH030.B0.2', 'HC_4.TH040.B0.2',
    'IPF_1.TD011.B1.1', 'IPF_1.TD011.B1.2', 'IPF_1.TD012.B2.1', 'IPF_1.TD012.B2.2', 'IPF_1.TD013.B3.1', 'IPF_1.TD013.B3.2',
    'IPF_2.TD021A.B1.1', 'IPF_2.TD021A.B1.2', 'IPF_2.TD021B.B2.1', 'IPF_2.TD021B.B2.2', 'IPF_2.TD022.B3.1', 'IPF_2.TD022.B3.2',
    'IPF_3.TD031.B1.2', 'IPF_3.TD032A.B2.2', 'IPF_3.TD032B.B3.2',
    'IPF_4.TD041A.B1.1', 'IPF_4.TD041A.B1.2', 'IPF_4.TD041B.B2.1', 'IPF_4.TD042.B3.1'
]

In [ ]:
ncols = 4
nrows = (len(order) + ncols - 1) // ncols
fig, axs = plt.subplots(nrows, ncols, figsize=(5 * ncols, 5 * nrows))
axs = axs.flatten()

for i, sample in enumerate(order):
    ad = adata[adata.obs["sample_name"] == sample].copy()
    sc.pl.umap(ad, color="leiden_0.4", size=65, legend_loc="on data", title=sample, ax=axs[i], show=False)

for j in range(i + 1, len(axs)):
    axs[j].axis("off")

plt.tight_layout()
#plt.savefig(f"{plot_directory}/umap_by_sample.png", dpi=600)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import squidpy as sq

genes = ["CTHRC1", "ADAMTS4"]
samples = ["HC_3.TH030.B0.2", "IPF_1.TD013.B3.2", "IPF_2.TD021B.B2.1", "IPF_3.TD032B.B3.2", "IPF_4.TD042.B3.1"]
genes = [g for g in genes if g in adata.var_names]

fig, axs = plt.subplots(2, 5, figsize=(25, 10), dpi=600)
axs = axs.flatten()

for j, gene in enumerate(genes):
    for i, sample in enumerate(samples):
        idx = j * 5 + i
        ad = adata[adata.obs["sample_name"] == sample]
        sq.pl.spatial_scatter(ad, color=gene, library_id=ad.obs["sample"].iloc[0], size=1.5,
                              cmap="rocket_r", ax=axs[idx], colorbar=False)
        im = axs[idx].collections[0]
        cb = fig.colorbar(im, ax=axs[idx], shrink=0.4, aspect=10, pad=0.01)
        axs[idx].set_title(f"{gene} in {sample}", fontsize=8)

for k in range(len(genes)*len(samples), len(axs)):
    axs[k].axis("off")

plt.tight_layout()
plt.savefig(f"{plot_directory}/{'_'.join(genes)}_spatial_expression.png", dpi=600, bbox_inches="tight")
plt.show()

In [ ]:
cluster2celltype = {
    '0': 'AT1/2',
    '1': 'MyoFBs',
    '2': 'CapEndothelial',
    '3': 'AlvMac',
    '4': 'DistalAirEpi',
    '5': 'Club/goblet',
    '6': 'AT2p',
    '7': 'SMCs',
    '8': 'Erythro',
    '9': 'Platelets',
    '10': 'Plasma/B',
    '11': 'AT2p2',
    '12': 'AT2(stress)',
    '13': 'SecretoryEpi',
    '14': 'IgAmucousSec',
    '15': 'Mucous/Ciliated',
    '16': 'AT1',
    '17': 'RareCiliated',
    '18': 'Erythro2',
    '19': 'Platelets2'
}

adata.obs['celltype'] = adata.obs['leiden_0.6'].map(cluster2celltype).astype('category')

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt

leiden_res = 'celltype'
cluster_fate_df = pd.DataFrame({
    'sample': adata.obs['sample_name'],
    'cluster': adata.obs[leiden_res]
})

cluster_frequencies = pd.crosstab(cluster_fate_df['sample'], cluster_fate_df['cluster'], normalize='index') * 100
cluster_frequencies = cluster_frequencies.reindex(order).fillna(0)

csv_file_path = os.path.join(plot_directory, f'Freq_per_treatment_group_{leiden_res}.csv')
cluster_frequencies.to_csv(csv_file_path)
print(f"Cluster frequencies saved to {csv_file_path}")

cluster_colors = adata.uns.get(f'{leiden_res}_colors', None)
if cluster_colors is None:
    import scanpy as sc
    cluster_colors = sc.pl.palettes.default_20

ax = cluster_frequencies.plot(
    kind='bar',
    stacked=True,
    color=cluster_colors[:cluster_frequencies.shape[1]],
    figsize=(10, 6)
)

plt.title(f'Cluster Frequency ({leiden_res})', fontsize=18, fontweight='bold')
plt.ylabel('Percentage (%)', fontsize=14)
plt.xlabel('Sample', fontsize=14)
plt.xticks(fontsize=12, rotation=45, ha='right')
plt.yticks(fontsize=10)
plt.legend(title=f'Cluster ({leiden_res})', title_fontsize=12, fontsize=10, bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
ax.grid(axis='y', linestyle='--', alpha=0.7)

pdf_path = os.path.join(plot_directory, f'Freq_per_treatment_group_{leiden_res}.pdf')
png_path = os.path.join(plot_directory, f'Freq_per_treatment_group_{leiden_res}.png')

plt.savefig(pdf_path, dpi=600, bbox_inches='tight')
plt.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')

print(f"Plot saved as PDF to {pdf_path}")
#print(f"Plot saved as PNG to {png_path}")

plt.show()

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu
import matplotlib

matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['font.family'] = 'Arial'
new_palette = {"control": "#4C72B0", "IPF": "#DD8452"}
def format_pvalue_scientific(p):
    if p == 0:
        return r"$\it{P}$ < 1 \times 10^{-300}"
    else:
        exponent = int(np.floor(np.log10(p)))
        mantissa = p / 10**exponent
        if exponent < -3:
            return rf"$\it{{P}} = {mantissa:.2f} \times 10^{{{exponent}}}$"
        else:
            return rf"$\it{{P}} = {p:.3g}$"

def plot_gene_expression_violin(adata, gene, save_path=None):
    df = adata.obs.copy()
    vals = adata[:, gene].X
    if hasattr(vals, "toarray"):
        vals = vals.toarray()
    df[f'{gene}_pos'] = (vals > 0).flatten().astype(int)
    df_grouped = df.groupby(['sample_name', 'condition'])[f'{gene}_pos'].sum().reset_index()

    ctrl = df_grouped.query("condition == 'control'")[f'{gene}_pos']
    ipf = df_grouped.query("condition == 'IPF'")[f'{gene}_pos']
    p = mannwhitneyu(ctrl, ipf, alternative='two-sided').pvalue

    star = 'ns'
    if p <= 0.0001: star = '****'
    elif p <= 0.001: star = '***'
    elif p <= 0.01: star = '**'
    elif p <= 0.05: star = '*'

    y_max = df_grouped[f'{gene}_pos'].max()

    fig, ax = plt.subplots(figsize=(4,6))
    sns.violinplot(data=df_grouped, x="condition", y=f'{gene}_pos', scale="width", inner=None,
               palette=new_palette, order=["control", "IPF"], ax=ax)
    sns.pointplot(data=df_grouped, x="condition", y=f'{gene}_pos', estimator="mean", errorbar=None,
              color='black', join=False, markers="D", scale=1.2, ax=ax)

    ax.set_ylabel(f"Spots expressing {gene}", fontsize=16, fontweight='bold')
    ax.set_xlabel("")
    ax.set_title(f"Expression of {gene}", fontsize=18, fontweight='bold')

    ax.set_xlim(-0.5, 1.5)
    ylim_low, ylim_high = ax.get_ylim()
    new_ylim_high = ylim_high * 1.25
    ax.set_ylim(ylim_low, new_ylim_high)

    star_y = new_ylim_high * 0.95
    ax.text(0.5, star_y, star, ha='center', fontsize=22, fontweight='bold', color='red' if star != 'ns' else 'black')

    p_text = format_pvalue_scientific(p)
    ax.text(0.5, star_y*0.88, p_text, ha='center', fontsize=14, fontstyle='italic', color='black')

    mean_ctrl = np.mean(ctrl)
    mean_ipf = np.mean(ipf)

    arrow_y = star_y * 1.1
    ax.plot([0.42, 0.58], [arrow_y, arrow_y], color='gray', linewidth=2)

    arrow_len = 0.15
    ylim_range = new_ylim_high - ylim_low
    head_len = ylim_range * 0.03
    head_wid = y_max * 0.03

    if mean_ipf > mean_ctrl:
        ax.arrow(0.44, arrow_y, arrow_len, 0, head_width=head_wid, head_length=head_len, fc='green', ec='green')
    else:
        ax.arrow(0.56, arrow_y, -arrow_len, 0, head_width=head_wid, head_length=head_len, fc='red', ec='red')

    sns.despine(trim=True)
    plt.tight_layout()

    if save_path is None:
        base_dir = "/Users/aly/Downloads/2024_Ali_scRNAseq/2025_Spatial_Fibrosis/2025_Squidpy_Spatial/Human/Analysis/Expression&Stastics"
        os.makedirs(base_dir, exist_ok=True)
        save_path = os.path.join(base_dir, f"{gene}_expression_violin.pdf")

    plt.savefig(save_path, dpi=300, facecolor='white')
    plt.show()
plot_gene_expression_violin(adata, 'ADAMTS4')

In [ ]:
import pandas as pd, seaborn as sns, matplotlib.pyplot as plt, scipy.stats as stats
import numpy as np

def format_pvalue_scientific(p):
    if p == 0:
        return r"$\it{P}$ < 1 \times 10^{-300}"
    else:
        exponent = int(np.floor(np.log10(p)))
        mantissa = p / 10**exponent
        if exponent < -3:
            return rf"$\it{{P}} = {mantissa:.2f} \times 10^{{{exponent}}}$"
        else:
            return rf"$\it{{P}} = {p:.3g}$"

df = adata.obs.copy()
df['CTHRC1_pos'] = (adata[:, 'CTHRC1'].X > 0).astype(int)
df['ADAMTS4_pos'] = (adata[:, 'ADAMTS4'].X > 0).astype(int)

df_grp = df.groupby(['condition', 'sample'])[['CTHRC1_pos', 'ADAMTS4_pos']].sum().reset_index()

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
for i, cond in enumerate(['control', 'IPF']):
    sub = df_grp[df_grp['condition'] == cond]
    sns.regplot(data=sub, x='CTHRC1_pos', y='ADAMTS4_pos', ax=axs[i], color='firebrick', scatter_kws={'s':80})
    r, p = stats.pearsonr(sub['CTHRC1_pos'], sub['ADAMTS4_pos'])
    p_text = format_pvalue_scientific(p)
    axs[i].set_title(f"{cond.upper()}\nPearson r = {r:.2f}, {p_text}", fontsize=12)
    axs[i].set_xlabel("Spots expressing CTHRC1")
    axs[i].set_ylabel("Spots expressing ADAMTS4")
    print(f"{cond.upper()}")
    print(f"r = {r:.2f}, p = {p_text}")

sns.set(font='Arial')
plt.tight_layout()
plt.savefig("/Users/aly/Downloads/2024_Ali_scRNAseq/2025_Spatial_Fibrosis/2025_Squidpy_Spatial/Human/Analysis/Expression&Stastics/CTHRC1_ADAMTS4_correlation_by_condition.pdf", dpi=300, facecolor='white')
plt.show()

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='condition', method='wilcoxon', use_raw=False)

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import scipy.cluster.hierarchy as sch
from matplotlib.font_manager import FontProperties


genes = list(set(adata.uns['rank_genes_groups']['names']['IPF'][:50].tolist() + 
                 adata.uns['rank_genes_groups']['names']['control'][:50].tolist()))

adata_sub = adata[:, genes].copy()
df = pd.DataFrame(adata_sub.X.toarray() if hasattr(adata_sub.X, "toarray") else adata_sub.X, 
                  index=adata_sub.obs['sample_name'], columns=genes)

# Scaling and ordering
df_mean = df.groupby(df.index).mean()
df_z = ((df_mean - df_mean.mean()) / df_mean.std()).clip(-1, 1)
sample_meta = adata_sub.obs[['sample_name','condition']].drop_duplicates().set_index('sample_name')
df_z = df_z.loc[sample_meta.sort_values(['condition','sample_name']).index]

# Clustering genes
col_order = sch.leaves_list(sch.linkage(df_z.T, method='average'))
df_z = df_z.iloc[:, col_order]

# Plotting
plt.rcParams['pdf.fonttype'] = 42
font_prop = FontProperties(family="Arial", style="italic", size=8)
fig, ax = plt.subplots(figsize=(6, 20))

hm = sns.heatmap(df_z.T, cmap=sns.diverging_palette(240, 10, as_cmap=True), center=0,
                 cbar=False, linewidths=0.5, linecolor='gray', ax=ax)

ax.set_yticks(range(len(df_z.columns)))
ax.set_yticklabels(df_z.columns, fontproperties=font_prop)
plt.xticks(rotation=45, ha='right', fontsize=5, fontname='Arial')
plt.title('Z-scaled Gene Expression per ST sample', fontsize=10, fontweight='bold', pad=5)

# Custom colorbar
cax = fig.add_axes([0.93, 0.4, 0.015, 0.2])
fig.colorbar(hm.collections[0], cax=cax, ticks=[-1, 0, 1]).set_label('Z-score', fontsize=10)

plt.savefig("heatmap_final.pdf", dpi=600, bbox_inches='tight')
plt.show()

In [ ]:
expr = adata[:, 'ADAMTS4'].X.toarray().flatten() if hasattr(adata[:, 'ADAMTS4'].X, "toarray") else adata[:, 'ADAMTS4'].X.flatten()
obs = adata.obs.copy()
obs['ADAMTS4_expr'] = expr
obs['ADAMTS4_pos'] = obs['ADAMTS4_expr'] > 0.2

df_pos = obs[obs['ADAMTS4_pos']].groupby('condition')['ADAMTS4_expr'].agg(['mean', 'median', 'count'])
df_pos['pct_positive'] = obs.groupby('condition')['ADAMTS4_pos'].mean() * 100

df_pos

In [ ]:
total_cells = adata.obs.groupby('condition').size()
adamts4_exp = adata[:, 'ADAMTS4'].X.toarray().flatten() if hasattr(adata[:, 'ADAMTS4'].X, 'toarray') else adata[:, 'ADAMTS4'].X.flatten()
adamts4_pos_cells = pd.Series(adamts4_exp > 0, index=adata.obs_names).groupby(adata.obs['condition']).sum()
pct_positive = (adamts4_pos_cells / total_cells) * 100
pct_positive

In [ ]:
expr = adata[:, 'ADAMTS4'].X.toarray().flatten() if hasattr(adata[:, 'ADAMTS4'].X, 'toarray') else adata[:, 'ADAMTS4'].X.flatten()
expr_log1p = np.log1p(expr)
mean_log_expr = pd.Series(expr_log1p, index=adata.obs_names).groupby(adata.obs['condition']).mean()
mean_log_expr

In [ ]:
expr = adata[:, 'ADAMTS4'].X.toarray().flatten() if hasattr(adata[:, 'ADAMTS4'].X, "toarray") else adata[:, 'ADAMTS4'].X.flatten()
conds = adata.obs['condition']

group_ipf = expr[conds == 'IPF']
group_control = expr[conds == 'control']

t_stat, p_val = stats.ttest_ind(group_ipf, group_control, equal_var=False)
p_val

In [ ]:
import scipy.stats as stats
expr = adata[:, 'ADAMTS4'].X.toarray().flatten() if hasattr(adata[:, 'ADAMTS4'].X, "toarray") else adata[:, 'ADAMTS4'].X.flatten()
conds = adata.obs['condition']
group_ipf = expr[conds == 'IPF']
group_control = expr[conds == 'control']
stat, p_val = stats.mannwhitneyu(group_ipf, group_control, alternative='two-sided')
p_val

In [ ]:
adata.write("/Users/aly/Downloads/2024_Ali_scRNAseq/2025_Spatial_Fibrosis/2025_Squidpy_Spatial/Human/Objects/2025_Human_IPF_Spatial_full.h5ad")